## load and prepare data

In [ ]:
%cd ../..
%matplotlib inline

import itertools
import os
import re
import subprocess
import tempfile

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import pyvista as pv
import seaborn as sns
import statsmodels.stats.contingency_tables as sm_contigency_tables
from scipy.stats import fisher_exact, mannwhitneyu, wilcoxon
from statsmodels.stats.proportion import proportion_confint
from statannotations.Annotator import Annotator
from statsmodels.stats.multitest import multipletests

import sys
sys.path.insert(0, 'evaluation_meldgraph')
from vol_eval_plots import (GROUP_3T, GROUP_3T_FLAIR, GROUP_7T_ADAPTED, GROUP_7T_ADAPTED_FLAIR,
                            GROUP_7T_DEFAULT, GROUP_3T_UNION_7T, latest_vol_eval_path, load_and_prepare_data,
                            filter_to_common_subjects, harmo_labels, main_group_hue_family,
                            harmo_saturation, _with_saturation, _with_value)
from vol_eval import FSAVERAGE_SYM_ANNOT_PATH


In [ ]:
eval_stats_df = load_and_prepare_data()

## sensitivity and specificity

In [ ]:
# for two given analysis groups (e.g. analysis_group='3T' and analysis_group='7T_adapted'), for each model, harmo, site_subj_id,
# create a new analysis group that is the logical OR of both
# for the columns recon_successful, tp_patient and tn_control
# 
# copy all other columns from the first condition, the following should be set to np.nan:
# input_path, prediction_path, ground_truth_path, pred_volume, gt_volume, intersection_volume,
# dice, num_gt_clusters, num_pred_clusters, tp_clusters, fp_clusters, max_cluster_dice
# and min_cluster_distance
#
# e.g.: for '3T' and '7T_adapted', the new analysis group will be '3T-union-7T_adapted'
#       if tp_patient is 1 for either condition, it will be 1 for the new condition

def create_logical_union_condition(df, condition1, condition2):
    cols_recomputed = ['analysis_group', 'recon_successful', 'tp_patient', 'tn_control', 'fp_clusters']

    merge_on = ['model', 'harmo', 'site_subj_id', 'source', 'site', 'category_FCD', 'category_3T_MR_negative', 'category_7T_MR_negative']

    columns_to_nan = [c for c in df.columns if c not in set(cols_recomputed + merge_on)]

    df_condition1 = df[df['analysis_group'] == condition1].copy()
    df_condition2 = df[df['analysis_group'] == condition2].copy()

    df_condition1[columns_to_nan] = np.nan
    df_condition2[columns_to_nan] = np.nan

    df_merged = pd.merge(
        df_condition1,
        df_condition2,
        on=merge_on + columns_to_nan,
        suffixes=(None, '_merged'),
        how='outer',
        validate='one_to_one'
    )

    if len(df_merged) != len(df_condition1) or len(df_merged) != len(df_condition2):
        raise ValueError(f"Merge of conditions {condition1} and {condition2} did not result in the same number of rows as the original conditions. "
                         f"len(df_condition1)={len(df_condition1)}, len(df_condition2)={len(df_condition2)}, len(df_merged)={len(df_merged)}")

    new_condition = f"{condition1}-union-{condition2}"

    def logical_or_with_nan(a, b):
        if pd.isna(a) and pd.isna(b):
            return np.nan
        if pd.isna(a):
            return b
        if pd.isna(b):
            return a
        return a or b

    def logical_and_with_nan(a, b):
        if pd.isna(a) and pd.isna(b):
            return np.nan
        if pd.isna(a):
            return b
        if pd.isna(b):
            return a
        return a and b

    # create new columns for the logical OR of recon_successful, tp_patient and tn_control
    df_merged['recon_successful'] = df_merged.apply(lambda row: logical_or_with_nan(row['recon_successful'], row['recon_successful_merged']), axis=1)
    df_merged['tp_patient'] = df_merged.apply(lambda row: logical_or_with_nan(row['tp_patient'], row['tp_patient_merged']), axis=1)
    df_merged['tn_control'] = df_merged.apply(lambda row: logical_and_with_nan(row['tn_control'], row['tn_control_merged']), axis=1)    
    df_merged['fp_clusters'] = df_merged['fp_clusters'] + df_merged['fp_clusters_merged']

    # overwrite the analysis_group column with the new condition
    df_merged['analysis_group'] = new_condition

    # drop all columns ending in _merged
    df_merged = df_merged[[col for col in df_merged.columns if not col.endswith('_merged')]]

    return df_merged

eval_stats_df = pd.concat([eval_stats_df, create_logical_union_condition(eval_stats_df, GROUP_3T, GROUP_7T_ADAPTED)])

In [ ]:
def metric_with_ci(df, outcome_col, alpha=0.05):
    # only patients should have tp_patient set and only controls should have tn_control set
    # otherwise this is np.nan and will be dropped
    df_filtered = df[~pd.isna(df[outcome_col])][['site_subj_id', outcome_col]].dropna(subset=[outcome_col])
    n = len(df_filtered)
    k = df_filtered[outcome_col].sum()
    p = k / n if n > 0 else np.nan
    ci_low, ci_high = proportion_confint(k, n, alpha=alpha, method='wilson') if n > 0 else (np.nan, np.nan)
    return df_filtered, n, k, p, ci_low, ci_high

def mean_with_ci(df, count_col, n_boot=10000, alpha=0.05, seed=0):
    values = df[count_col].dropna().to_numpy()
    n = len(values)
    mean = values.mean() if n > 0 else np.nan
    if n == 0:
        return n, mean, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot_means = rng.choice(values, size=(n_boot, n), replace=True).mean(axis=1)
    ci_low, ci_high = np.percentile(boot_means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return n, mean, ci_low, ci_high

def mcnemar_paired(df1, df2, outcome_col):
    # match subjects present in both conditions to get paired binary outcomes
    merged = pd.merge(df1, df2,
                      on='site_subj_id', 
                      suffixes=('_1', '_2'),
                      how='inner',
                      validate='one_to_one')
    # if subjects were dropped due to the merge, print a warning
    if len(merged) < min(len(df1), len(df2)):
        print(f"Warning: {len(df1) - len(merged)} subjects in df1 and {len(df2) - len(merged)} subjects in df2 were dropped due to no match on site_subj_id.")

    outcome_col_1, outcome_col_2 = f'{outcome_col}_1', f'{outcome_col}_2'
    merged = merged.dropna(subset=[outcome_col_1, outcome_col_2])
    both = int(((merged[outcome_col_1] == 1) & (merged[outcome_col_2] == 1)).sum())
    only1 = int(((merged[outcome_col_1] == 1) & (merged[outcome_col_2] == 0)).sum())
    only2 = int(((merged[outcome_col_1] == 0) & (merged[outcome_col_2] == 1)).sum())
    neither = int(((merged[outcome_col_1] == 0) & (merged[outcome_col_2] == 0)).sum())
    table = [[both, only1], [only2, neither]]
    result = sm_contigency_tables.mcnemar(table, exact=True)
    return merged.shape[0], table, result.statistic, result.pvalue

In [ ]:
def build_performance_table(eval_stats_df, analysis_groups, harmo_conditions):
    # sensitivity/specificity (+ 95%-CI, and a formatted "_display" string of both) for every
    # (harmo, analysis_group) combination in the given lists
    eval_stats_df = filter_to_common_subjects(eval_stats_df, analysis_groups, harmo_conditions)

    # we need to compute these before filtering to only include recon_successful==1
    failed_recons = {}
    for harmo, analysis_group in itertools.product(harmo_conditions, analysis_groups):
        failed_recons[(harmo, analysis_group, 'nsubjects')] = ((eval_stats_df['harmo'] == harmo) & 
                                                     (eval_stats_df['analysis_group'] == analysis_group)).sum()
        failed_recons[(harmo, analysis_group, 'all')] = ((eval_stats_df['harmo'] == harmo) & 
                                              (eval_stats_df['analysis_group'] == analysis_group) & 
                                              (eval_stats_df['recon_successful'] == 0)).sum()
        failed_recons[(harmo, analysis_group, 'patients')] = ((eval_stats_df['harmo'] == harmo) &
                                                   (eval_stats_df['analysis_group'] == analysis_group) &
                                                   (eval_stats_df['group'] == 'patient') &
                                                   (eval_stats_df['recon_successful'] == 0)).sum()
        failed_recons[(harmo, analysis_group, 'controls')] = ((eval_stats_df['harmo'] == harmo) &
                                                   (eval_stats_df['analysis_group'] == analysis_group) &
                                                   (eval_stats_df['group'] == 'control') &
                                                   (eval_stats_df['recon_successful'] == 0)).sum()

    # also keep a copy that retains subjects with failed reconstructions, with a binary recon_failed
    # column, used below for both the raw counts and the Wilson CI on the failed-recon rate
    eval_stats_df_recon = eval_stats_df.copy()
    eval_stats_df_recon['recon_failed'] = (eval_stats_df_recon['recon_successful'] == 0).astype(int)

    # now filter to only include subjects that were successfully reconstructed (recon_successful==1)
    eval_stats_df = eval_stats_df[eval_stats_df['recon_successful'] == 1]
    eval_stats_df = filter_to_common_subjects(eval_stats_df, analysis_groups, harmo_conditions)

    def format_metric(value, ci_low, ci_high):
        if pd.isna(value):
            return 'n/a'
        return f"{value * 100:.1f}% (95% CI: {ci_low * 100:.1f}-{ci_high * 100:.1f})"

    rows = []
    for harmo, analysis_group in itertools.product(harmo_conditions, analysis_groups):
        df_condition = eval_stats_df[(eval_stats_df['harmo'] == harmo) & (eval_stats_df['analysis_group'] == analysis_group)]
        df_condition_mrn = df_condition[df_condition['category_3T_MR_negative'] == 1]
        df_condition_histo = df_condition[(df_condition['confirmed_histology'] == 1)]
        df_condition_recon = eval_stats_df_recon[(eval_stats_df_recon['harmo'] == harmo) & (eval_stats_df_recon['analysis_group'] == analysis_group)]

        _, n_patients, k_patients, sensitivity, sens_ci_low, sens_ci_high = metric_with_ci(df_condition, 'tp_patient')
        _, n_mrnpatients, k_mrnpatients, sensitivity_mrn, sens_mrn_ci_low, sens_mrn_ci_high = metric_with_ci(df_condition_mrn, 'tp_patient')
        _, n_histo, k_histo, sensitivity_histo, sens_histo_ci_low, sens_histo_ci_high = metric_with_ci(df_condition_histo, 'tp_patient')
        _, n_controls, k_controls, specificity, spec_ci_low, spec_ci_high = metric_with_ci(df_condition, 'tn_control')
        _, n_recon, k_recon, failed_recon_rate, recon_ci_low, recon_ci_high = metric_with_ci(df_condition_recon, 'recon_failed')
        n_fp_clusters, mean_no_fp_clusters, fp_ci_low, fp_ci_high = mean_with_ci(df_condition, 'fp_clusters')

        rows.append({
            'harmo': harmo,
            'analysis_group': analysis_group,
            'n_subjects': failed_recons[(harmo, analysis_group, 'nsubjects')],
            'n_subjects_failed_recon': failed_recons[(harmo, analysis_group, 'all')],
            'n_patients_failed_recon': failed_recons[(harmo, analysis_group, 'patients')],
            'n_controls_failed_recon': failed_recons[(harmo, analysis_group, 'controls')],
            'failed_recon_rate': failed_recon_rate,
            'failed_recon_rate_ci_low': recon_ci_low,
            'failed_recon_rate_ci_high': recon_ci_high,
            'failed_recon_rate_display': format_metric(failed_recon_rate, recon_ci_low, recon_ci_high),
            'n_patients': n_patients,
            'k_patients': k_patients,
            'sensitivity': sensitivity,
            'sensitivity_ci_low': sens_ci_low,
            'sensitivity_ci_high': sens_ci_high,
            'sensitivity_display': format_metric(sensitivity, sens_ci_low, sens_ci_high),
            'n_mrnpatients': n_mrnpatients,
            'k_mrnpatients': k_mrnpatients,
            'sensitivity_mrn': sensitivity_mrn,
            'sensitivity_mrn_ci_low': sens_mrn_ci_low,
            'sensitivity_mrn_ci_high': sens_mrn_ci_high,
            'sensitivity_mrn_display': format_metric(sensitivity_mrn, sens_mrn_ci_low, sens_mrn_ci_high),
            'n_patients_histo': n_histo,
            'k_patients_histo': k_histo,
            'sensitivity_histo': sensitivity_histo,
            'sensitivity_histo_ci_low': sens_histo_ci_low,
            'sensitivity_histo_ci_high': sens_histo_ci_high,
            'sensitivity_histo_display': format_metric(sensitivity_histo, sens_histo_ci_low, sens_histo_ci_high),
            'n_controls': n_controls,
            'k_controls': k_controls,
            'specificity': specificity,
            'specificity_ci_low': spec_ci_low,
            'specificity_ci_high': spec_ci_high,
            'specificity_display': format_metric(specificity, spec_ci_low, spec_ci_high),
            'n_fp_clusters': n_fp_clusters,
            'mean_no_fp_clusters': mean_no_fp_clusters,
            'mean_no_fp_clusters_ci_low': fp_ci_low,
            'mean_no_fp_clusters_ci_high': fp_ci_high,
            'mean_no_fp_clusters_display': f"{mean_no_fp_clusters:.2f} (95% CI: {fp_ci_low:.2f}-{fp_ci_high:.2f})"
        })
    return pd.DataFrame(rows)

In [ ]:
performance_table = build_performance_table(eval_stats_df, [GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED], ['noharmo', 'harmo'])
performance_table = pd.concat([performance_table, build_performance_table(eval_stats_df, [GROUP_3T_FLAIR], ['harmo'])], axis=0).reset_index(drop=True)
performance_table = pd.concat([performance_table, build_performance_table(eval_stats_df, [GROUP_7T_ADAPTED_FLAIR], ['harmo'])], axis=0).reset_index(drop=True)
performance_table = pd.concat([performance_table, build_performance_table(eval_stats_df, [GROUP_3T_UNION_7T], ['harmo'])], axis=0).reset_index(drop=True)
# show only some columns of interest, and sort
performance_table = performance_table[['analysis_group',
                                       'harmo', 
                                       'n_subjects',
                                       'n_subjects_failed_recon',
                                       'failed_recon_rate_display',
                                       'n_patients',
                                       'k_patients',
                                       'sensitivity_display',
                                       'n_mrnpatients',
                                       'k_mrnpatients',
                                       'sensitivity_mrn_display',
                                       'n_patients_histo',
                                       'k_patients_histo',
                                       'sensitivity_histo_display',
                                       'n_controls',
                                       'k_controls',
                                       'specificity_display',
                                       'mean_no_fp_clusters_display']]
# sort order for analysis_group: 3T, 7T_default, 7T_adapted, 3T_FLAIR, 7T_adapted_FLAIR, 3T-UNION-7T_adapted
# sort order for harmo: noharmo, harmo
performance_table.sort_values(by=['analysis_group', 'harmo'], key=lambda x: x.map({GROUP_3T: 0, GROUP_7T_DEFAULT: 1, GROUP_7T_ADAPTED: 2, GROUP_3T_FLAIR: 3, GROUP_7T_ADAPTED_FLAIR: 4, GROUP_3T_UNION_7T: 5, 'noharmo': 0, 'harmo': 1}), inplace=True)
performance_table = performance_table.T
performance_table

In [ ]:
def plot_sensitivity_specificity(eval_stats_df, 
                                 analysis_groups, 
                                 harmo_conditions, 
                                 group_hue_family, 
                                 group_tick_labels=None, 
                                 figsize=(6, 9),
                                 correction_method='fdr_bh',
                                 panels=None,
                                 axes=None,
                                 annotate_pairs=True):
    """
    Build bar plot with (sensitivity / sensitivity in MR-negative patients / specificity etc.)
    with 95%-CI error bars, for the given analysis groups crossed with the given harmo conditions.

    harmo_conditions=['noharmo', 'harmo'] draws paired bars per analysis group (with noharmo-vs-harmo
    McNemar brackets in addition to the pairwise, harmonized-only, across-group ones) and a second
    row of per-bar harmo x-tick labels; harmo_conditions=['harmo'] draws a single bar per analysis
    group with only the pairwise, across-group McNemar brackets.

    correction_method is passed to statsmodels' multipletests, e.g. 'fdr_bh' or 'bonferroni'.

    panels restricts the plot to the given metrics (e.g. ['sensitivity', 'specificity']), which are
    then lettered A, B, ... in that order; None draws all of them.

    axes draws into the given column of axes (one per panel, e.g. from subfigure.subplots(sharex=True))
    instead of a new figure; the layout is then left to the caller.

    annotate_pairs=False leaves out the significance brackets of the pairwise tests; their
    (corrected) p-values are still printed.
    """
    # get the performance table for the given analysis groups and harmo conditions
    # this internally filters for matched subjects across all (harmo, analysis_group) 
    # combinations and recon_successful==1 and also calculates the number 
    # of subjects that failed recon
    performance_table = build_performance_table(eval_stats_df, analysis_groups, harmo_conditions)

    # filter the eval_stats_df to only include subjects present in all 
    # (harmo, analysis_group) combinations 
    eval_stats_df_recon = filter_to_common_subjects(eval_stats_df, analysis_groups, harmo_conditions).copy()
    eval_stats_df_recon['recon_failed'] = (eval_stats_df_recon['recon_successful'] == 0).astype(int)

    # then also filter to only include subjects that were successfully reconstructed (recon_successful==1)
    # and recon_successful==1 (needed for the McNemar tests)
    eval_stats_df = filter_to_common_subjects(eval_stats_df[eval_stats_df['recon_successful'] == 1],
                                              analysis_groups,
                                              harmo_conditions)

    hue_order = [harmo_labels[harmo] for harmo in harmo_conditions]

    metrics = [
        ('failed_recons', 'Failed reconstructions'),
        ('sensitivity', 'Sensitivity (all patients)'),
        ('sensitivity_mrn', 'Sensitivity (3T MRI-negative patients)'),
        ('sensitivity_histo', 'Sensitivity (histologically verified FCD)'),
        ('specificity', 'Specificity (healthy controls)'),
        ('mean_no_fp_clusters', 'Mean false positives per subject (patients and controls)'),
    ]
    if panels is not None:
        metrics = [(metric_col, title) for metric_col, title in metrics if metric_col in panels]
    metrics = [(metric_col, f'{chr(ord("A") + i)} | {title}') for i, (metric_col, title) in enumerate(metrics)]

    if axes is None:
        fig, axes = plt.subplots(nrows=len(metrics), ncols=1, figsize=figsize, sharex=True,
                                 gridspec_kw={'height_ratios': [0.15 if metric_col == 'failed_recons' else 1
                                                                for metric_col, _ in metrics]})
        layout_figure = True
    else:
        fig = axes[0].figure
        layout_figure = False

    # raw count columns (built by build_performance_table) backing each metric's bar
    count_columns = {
        'failed_recons': ('n_subjects_failed_recon', 'n_subjects'),
        'sensitivity': ('k_patients', 'n_patients'),
        'sensitivity_mrn': ('k_mrnpatients', 'n_mrnpatients'),
        'sensitivity_histo': ('k_patients_histo', 'n_patients_histo'),
        'specificity': ('k_controls', 'n_controls'),
    }

    for ax, (metric_col, title) in zip(axes, metrics):
        if metric_col == 'failed_recons':
            # this is a special case, we only show the raw counts, no actual 
            # bar plot
            metric_df = performance_table[['analysis_group', 'harmo', 'n_subjects_failed_recon', 'n_subjects']].copy()
            metric_df['Harmonization'] = metric_df['harmo'].map(harmo_labels)
            metric_df['value'] = metric_df['n_subjects_failed_recon']

            # draw an invisible barplot purely to reuse the same x-positions/dodge-grouping as the
            # other subplots (sharex=True); bar heights equal the actual failed-recon counts so the
            # McNemar brackets below are stacked at sensible heights
            sns.barplot(data=metric_df, x='analysis_group', y='value', hue='Harmonization',
                        order=analysis_groups, hue_order=hue_order, palette=['none'] * len(hue_order),
                        errorbar=None, legend=False, ax=ax)

            # fix the y-range before annotating so the brackets are placed relative to it, not the
            # auto-scaled range (mirrors the other 3 panels)
            ylim_top = max(metric_df['n_subjects_failed_recon'].max(), 1) * 1.6
            ax.set_ylim(0, ylim_top)

            # n_subjects_failed_recon/n_subjects doesn't depend on harmo (reconstruction success is
            # independent of harmonization), so annotate once per analysis group, centered across that
            # group's (possibly dodged) bar positions, using the first harmo condition's values
            sub = metric_df[metric_df['harmo'] == harmo_conditions[0]].set_index('analysis_group').loc[analysis_groups]
            text_y = ylim_top * 0.05
            for group_idx, analysis_group in enumerate(analysis_groups):
                group_x_positions = [container[group_idx].get_x() + container[group_idx].get_width() / 2
                                    for container in ax.containers]
                x_pos = np.mean(group_x_positions)
                failed, total = sub.loc[analysis_group, ['n_subjects_failed_recon', 'n_subjects']]
                ax.text(x_pos, text_y, f'{int(failed)}/{int(total)}',
                        ha='center', va='bottom', fontsize=10, color='black')

            for container in ax.containers:
                for patch in container:
                    patch.set_visible(False)

            # McNemar tests for recon failure; use eval_stats_df_recon, which (unlike eval_stats_df
            # used by the other 3 panels) retains subjects whose reconstruction failed
            # display bracket only if p<0.05
            outcome_col = 'recon_failed'
            pairs, pvalues = [], []

            for group_a, group_b in itertools.combinations(analysis_groups, 2):
                df_a = eval_stats_df_recon[(eval_stats_df_recon['harmo'] == 'harmo') & (eval_stats_df_recon['analysis_group'] == group_a)]
                df_b = eval_stats_df_recon[(eval_stats_df_recon['harmo'] == 'harmo') & (eval_stats_df_recon['analysis_group'] == group_b)]
                _, _, _, pvalue = mcnemar_paired(df_a, df_b, outcome_col)
                if pvalue < 0.05:
                    pairs.append(((group_a, harmo_labels['harmo']), (group_b, harmo_labels['harmo'])))
                    pvalues.append(pvalue)

            if annotate_pairs and len(pairs) > 0:
                annotator = Annotator(ax, pairs, plot='barplot', data=metric_df, x='analysis_group', y='value',
                                        hue='Harmonization', order=analysis_groups, hue_order=hue_order, verbose=0)
                annotator.configure(test=None,
                                    text_format='star',
                                    loc='inside',
                                    line_offset_to_group=0.01,
                                    line_offset=0.005,
                                    line_height=0.02,
                                    fontsize=9,
                                    color='gray')
                annotator.pvalue_format.pvalue_thresholds = [
                    (0.05, '*'),
                    (1, 'ns')
                ]
                annotator.set_pvalues(pvalues)
                annotator.annotate()

            ax.set_yticks([])
            ax.set_ylabel('')
            ax.set_xlabel('')
            ax.set_title(title, fontsize=10, ha='left', x=0.0, y=1.02, fontweight='bold')
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.tick_params(left=False)
            ax.tick_params(axis='x', which='both', length=0)

            continue
        elif metric_col == 'mean_no_fp_clusters':
            # not a proportion, so this gets its own bar plot instead of reusing the
            # percentage scaling used for sensitivity/specificity below
            metric_df = performance_table[['analysis_group', 'harmo', 'mean_no_fp_clusters',
                                           'mean_no_fp_clusters_ci_low', 'mean_no_fp_clusters_ci_high',
                                           'n_fp_clusters']].copy()
            metric_df['Harmonization'] = metric_df['harmo'].map(harmo_labels)
            metric_df['value'] = metric_df['mean_no_fp_clusters']
            metric_df['ci_low'] = metric_df['mean_no_fp_clusters_ci_low']
            metric_df['ci_high'] = metric_df['mean_no_fp_clusters_ci_high']

            sns.barplot(data=metric_df, x='analysis_group', y='value', hue='Harmonization',
                        order=analysis_groups, hue_order=hue_order, palette=['lightgray'] * len(hue_order),
                        errorbar=None, legend=False, ax=ax)

            # fix the y-range before annotating so the "n=" labels are placed relative to it, not
            # the auto-scaled range (mirrors the other panels); guard against an all-NaN metric_df
            # (e.g. for UNION conditions, where fp_clusters is not meaningful)
            ylim_top = np.nanmax([metric_df['ci_high'].max(skipna=True), 1]) * 1.3

            for container, harmo in zip(ax.containers, harmo_conditions):
                sub = metric_df[metric_df['harmo'] == harmo].set_index('analysis_group').loc[analysis_groups]
                for patch, analysis_group in zip(container, analysis_groups):
                    color = _with_saturation(group_hue_family[analysis_group], harmo_saturation[harmo])
                    patch.set_facecolor(color)
                    x_pos = patch.get_x() + patch.get_width() / 2
                    value, ci_low, ci_high, n = sub.loc[analysis_group, ['value', 'ci_low', 'ci_high', 'n_fp_clusters']]
                    if pd.isna(value):
                        continue
                    height = patch.get_height()
                    yerr = [[value - ci_low], [ci_high - value]]
                    ax.errorbar([x_pos], [height], yerr=yerr, fmt='none', ecolor=_with_value(color, 0.6), capsize=3, linewidth=1)
                    ax.text(x_pos, 0, f'{sub.loc[analysis_group, "value"]:.1f}', ha='center', va='bottom', fontsize=8, color='black')

            ax.set_ylim(0, ylim_top)
            ax.set_title(title, fontsize=10, ha='left', x=0.0, y=1.02, fontweight='bold')
            ax.set_ylabel('Mean count')
            ax.set_xlabel('')
            ax.tick_params(axis='x', which='both', length=0)
            ax.grid(axis='y', color='black', alpha=0.15, linewidth=0.8, zorder=0)
            ax.set_axisbelow(True)
            for spine in ['top', 'right']:
                ax.spines[spine].set_visible(False)

            # pairwise Wilcoxon signed rank tests for the mean_no_fp_clusters metric; the two conditions
            # are merged on site_subj_id (verifying a 1:1 match)
            pairs, pvalues = [], []
            if len(harmo_conditions) > 1:
                # noharmo vs harmo, within each analysis group
                for analysis_group in analysis_groups:
                    df_noharmo = eval_stats_df[(eval_stats_df['harmo'] == 'noharmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    df_harmo = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    merged = pd.merge(df_noharmo[['site_subj_id', 'fp_clusters']], df_harmo[['site_subj_id', 'fp_clusters']],
                                      on='site_subj_id', suffixes=('_noharmo', '_harmo'), how='inner', validate='one_to_one')
                    _, pvalue = wilcoxon(merged['fp_clusters_noharmo'], merged['fp_clusters_harmo'])
                    pairs.append(((analysis_group, harmo_labels['noharmo']), (analysis_group, harmo_labels['harmo'])))
                    pvalues.append(pvalue)

            # pairwise Wilcoxon tests between the analysis groups, within the harmonized group; their
            # wider x-range makes statannotations auto-stack them above the noharmo-vs-harmo brackets
            for group_a, group_b in itertools.combinations(analysis_groups, 2):
                df_a = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_a)]
                df_b = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_b)]
                merged = pd.merge(df_a[['site_subj_id', 'fp_clusters']], df_b[['site_subj_id', 'fp_clusters']],
                                  on='site_subj_id', suffixes=('_a', '_b'), how='inner', validate='one_to_one')
                _, pvalue = wilcoxon(merged['fp_clusters_a'], merged['fp_clusters_b'])
                pairs.append(((group_a, harmo_labels['harmo']), (group_b, harmo_labels['harmo'])))
                pvalues.append(pvalue)

        else:
            k_col, n_col = count_columns[metric_col]
            metric_df = performance_table[['analysis_group', 'harmo', metric_col, f'{metric_col}_ci_low', f'{metric_col}_ci_high', k_col, n_col]].copy()
            metric_df['Harmonization'] = metric_df['harmo'].map(harmo_labels)
            metric_df['value'] = metric_df[metric_col] * 100
            metric_df['ci_low'] = metric_df[f'{metric_col}_ci_low'] * 100
            metric_df['ci_high'] = metric_df[f'{metric_col}_ci_high'] * 100

            sns.barplot(data=metric_df, x='analysis_group', y='value', hue='Harmonization',
                        order=analysis_groups, hue_order=hue_order, palette=['lightgray'] * len(hue_order),
                        errorbar=None, legend=False, ax=ax)

            # recolor each bar by (analysis_group, harmo) and overlay asymmetric 95%-CI error bars in the same color
            for container, harmo in zip(ax.containers, harmo_conditions):
                sub = metric_df[metric_df['harmo'] == harmo].set_index('analysis_group').loc[analysis_groups]
                for patch, analysis_group in zip(container, analysis_groups):
                    color = _with_saturation(group_hue_family[analysis_group], harmo_saturation[harmo])
                    patch.set_facecolor(color)
                    x_pos = patch.get_x() + patch.get_width() / 2
                    height = patch.get_height()
                    value, ci_low, ci_high, k, n = sub.loc[analysis_group, ['value', 'ci_low', 'ci_high', k_col, n_col]]
                    yerr = [[value - ci_low], [ci_high - value]]
                    ax.errorbar([x_pos], [height], yerr=yerr, fmt='none', ecolor=_with_value(color, 0.6), capsize=3, linewidth=1)
                    # raw counts behind the percentage (e.g. true positives / patients), near the bar's base
                    ax.text(x_pos, 2, f'{int(k)}/{int(n)}', ha='center', va='bottom', fontsize=8, color='black')

            # fix the y-range before annotating so the brackets are placed relative to it, not the auto-scaled range
            ax.set_ylim(0, 90)

            outcome_col = 'tn_control' if metric_col == 'specificity' else 'tp_patient'
            pairs, pvalues = [], []

            if len(harmo_conditions) > 1:
                # McNemar test for noharmo vs harmo, paired by site_subj_id, within each analysis group
                # mcnemar_paired() merged on site_subj_id and verifies a 1:1 match
                for analysis_group in analysis_groups:
                    df_noharmo = eval_stats_df[(eval_stats_df['harmo'] == 'noharmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    df_harmo = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == analysis_group)]
                    if metric_col == 'sensitivity_mrn':
                        df_noharmo = df_noharmo[df_noharmo['category_3T_MR_negative'] == 1]
                        df_harmo = df_harmo[df_harmo['category_3T_MR_negative'] == 1]
                    elif metric_col == 'sensitivity_histo':
                        df_noharmo = df_noharmo[df_noharmo['confirmed_histology'] == 1]
                        df_harmo = df_harmo[df_harmo['confirmed_histology'] == 1]
                    _, _, _, pvalue = mcnemar_paired(df_noharmo, df_harmo, outcome_col)
                    pairs.append(((analysis_group, harmo_labels['noharmo']), (analysis_group, harmo_labels['harmo'])))
                    pvalues.append(pvalue)

            # pairwise McNemar tests between the analysis groups, within the harmonized group; their
            # wider x-range makes statannotations auto-stack them above the noharmo-vs-harmo brackets
            for group_a, group_b in itertools.combinations(analysis_groups, 2):
                df_a = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_a)]
                df_b = eval_stats_df[(eval_stats_df['harmo'] == 'harmo') & (eval_stats_df['analysis_group'] == group_b)]
                if metric_col == 'sensitivity_mrn':
                    df_a = df_a[df_a['category_3T_MR_negative'] == 1]
                    df_b = df_b[df_b['category_3T_MR_negative'] == 1]
                elif metric_col == 'sensitivity_histo':
                    df_a = df_a[df_a['confirmed_histology'] == 1]
                    df_b = df_b[df_b['confirmed_histology'] == 1]
                _, _, _, pvalue = mcnemar_paired(df_a, df_b, outcome_col)
                pairs.append(((group_a, harmo_labels['harmo']), (group_b, harmo_labels['harmo'])))
                pvalues.append(pvalue)

        # correct p-values for multiple testing
        _, pvalues, _, _ = multipletests(pvalues, alpha=0.05, method=correction_method)

        for pair, pvalue in zip(pairs, pvalues):
            print(f"{pair[0]} vs {pair[1]}: p={pvalue:.4f}")

        pairs_filtered, pvalues_filtered = [], []
        for pair, pvalue in zip(pairs, pvalues):
            if pvalue < 0.05:
                pairs_filtered.append(pair)
                pvalues_filtered.append(pvalue)
        pairs, pvalues = pairs_filtered, pvalues_filtered

        if annotate_pairs and len(pairs) > 0: # if any p-values survive the correction, annotate them
            annotator = Annotator(ax, pairs, plot='barplot', data=metric_df, x='analysis_group', y='value',
                                    hue='Harmonization', order=analysis_groups, hue_order=hue_order, verbose=0)
            annotator.configure(test=None,
                                text_format='star',
                                show_test_name=False,
                                loc='inside',
                                line_offset_to_group=0.01,
                                line_offset=0.005,
                                line_height=0.02,
                                fontsize=9,
                                color='gray')
            annotator.pvalue_format.pvalue_thresholds = [
                (0.05, '*'),
                (1, 'ns')
            ]
            annotator.set_pvalues(pvalues)
            annotator.annotate()



        if metric_col in ['sensitivity', 'sensitivity_mrn', 'sensitivity_histo', 'specificity']:
            ax.set_ylabel('%')
            # cap the visible ticks/gridlines/y-spine at 90 even though ylim may extend higher
            ax.set_yticks(np.arange(0, 91, 20))
            ax.spines['left'].set_bounds(0, 90)
            ax.set_title(title, fontsize=10, ha='left', x=0.0, y=1.02, fontweight='bold')
            ax.set_xlabel('')
            ax.tick_params(axis='x', which='both', length=0)
            # set_ylabel() centers on the full (possibly annotation-expanded) ylim by default; re-anchor
            # it to the middle of the visible 0-90 band instead. This takes over from matplotlib's
            # automatic label placement, so the x-offset is now a fixed axes-fraction value rather than
            # one that adapts to tick-label width
            ax.yaxis.set_label_coords(-0.06, 45, transform=mtransforms.blended_transform_factory(ax.transAxes, ax.transData))
            ax.grid(axis='y', color='black', alpha=0.15, linewidth=0.8, zorder=0)
            ax.set_axisbelow(True)
            for spine in ['top', 'right']:
                ax.spines[spine].set_visible(False)
        elif metric_col == 'mean_no_fp_clusters':
            pass



    axes[-1].set_xticks(range(len(analysis_groups)))
    axes[-1].set_xticklabels([(group_tick_labels or {}).get(analysis_group, analysis_group) for analysis_group in analysis_groups])

    if len(harmo_conditions) > 1:
        # add a second row of x-axis labels naming the harmo condition of each individual bar
        minor_positions, minor_labels = [], []
        for container, harmo in zip(axes[-1].containers, harmo_conditions):
            for patch in container:
                minor_positions.append(patch.get_x() + patch.get_width() / 2)
                minor_labels.append(harmo_labels[harmo])

        axes[-1].set_xticks(minor_positions, minor=True)
        axes[-1].set_xticklabels(minor_labels, minor=True, fontsize=8, rotation=45, ha='right', rotation_mode='anchor')
        axes[-1].tick_params(axis='x', which='minor', pad=2, length=0)
        axes[-1].tick_params(axis='x', which='major', pad=40, length=0, labelsize=10)

    if layout_figure:
        fig.tight_layout(h_pad=1.5)
    return fig, axes, performance_table

### Figure 2

In [ ]:
fig, axes, performance_table = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED],
    harmo_conditions=['noharmo', 'harmo'],
    group_hue_family=main_group_hue_family,
)

### list cases as example plot candidates

In [ ]:
# list possible subjects to plot
eval_stats_df_3T7T = eval_stats_df[(eval_stats_df['analysis_group'].isin([GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED])) & (eval_stats_df['harmo'] == 'harmo')]
eval_stats_df_3T7T = filter_to_common_subjects(eval_stats_df_3T7T, [GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED], ['harmo'])
eval_stats_df_3T7T_patients = eval_stats_df_3T7T.pivot(index=['site_subj_id', 'confirmed_histology'], columns='analysis_group', values='tp_patient').dropna().reset_index()
eval_stats_df_3T7T_controls = eval_stats_df_3T7T.pivot(index=['site_subj_id', 'confirmed_histology'], columns='analysis_group', values='tn_control').dropna().reset_index()
print(eval_stats_df_3T7T_patients.head())
print(eval_stats_df_3T7T_controls.head())

In [ ]:
# patients positive at 3T and 7T adapted
eval_stats_df_3T7T_patients[(eval_stats_df_3T7T_patients[GROUP_3T] == 1) & (eval_stats_df_3T7T_patients[GROUP_7T_ADAPTED] == 1)]

In [ ]:
# patients negative at 3T and positive at either 7T adapted or 7T default
eval_stats_df_3T7T_patients[(eval_stats_df_3T7T_patients[GROUP_3T] == 0) & ((eval_stats_df_3T7T_patients[GROUP_7T_DEFAULT] == 1) | (eval_stats_df_3T7T_patients[GROUP_7T_ADAPTED] == 1))]

In [ ]:
# patients positive at 3T and negative at either 7T adapted or 7T default
eval_stats_df_3T7T_patients[(eval_stats_df_3T7T_patients[GROUP_3T] == 1) & ((eval_stats_df_3T7T_patients[GROUP_7T_DEFAULT] == 0) | (eval_stats_df_3T7T_patients[GROUP_7T_ADAPTED] == 0))]

In [ ]:
# patients negative at all conditions but histopathologically verified
eval_stats_df_3T7T_patients[((~eval_stats_df_3T7T_patients[[GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED]].any(axis=1)) & (eval_stats_df_3T7T_patients['confirmed_histology'] == 1))]

In [ ]:
# controls negative at 3T but with false positive at either 7T adapted or 7T default
eval_stats_df_3T7T_controls[(eval_stats_df_3T7T_controls[GROUP_3T] == 1) & ((eval_stats_df_3T7T_controls[GROUP_7T_DEFAULT] == 0) | (eval_stats_df_3T7T_controls[GROUP_7T_ADAPTED] == 0))]

In [ ]:
# controls positive in different conditions 
eval_stats_df_3T7T_controls[(eval_stats_df_3T7T_controls[GROUP_3T] == 0) & (eval_stats_df_3T7T_controls[GROUP_7T_DEFAULT] == 0) & (eval_stats_df_3T7T_controls[GROUP_7T_ADAPTED] == 1)]

### union/combined 3T + 7T

### Supplementary Figure 3

In [ ]:
fig_union, axes_union, performance_table_union = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_7T_ADAPTED, GROUP_3T_UNION_7T],
    harmo_conditions=['harmo'],
    panels=['failed_recons', 'sensitivity', 'sensitivity_mrn', 'specificity', 'mean_no_fp_clusters'],
    group_hue_family={GROUP_3T: "#a9c8f0", GROUP_7T_ADAPTED: '#f4b78a', GROUP_3T_UNION_7T: '#a8dab5'},
    group_tick_labels={GROUP_3T_UNION_7T: '3T + 7T adapted'},
)

### Supplementary Figure 4

In [ ]:
fig_3tflair, axes_3tflair, performance_table_3tflair = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_3T_FLAIR],
    harmo_conditions=['harmo'],
    panels=['failed_recons', 'sensitivity', 'sensitivity_mrn', 'specificity', 'mean_no_fp_clusters'],
    group_hue_family={GROUP_3T: "#a9c8f0", GROUP_3T_FLAIR: _with_value('#a9c8f0', 0.8)},
    group_tick_labels={GROUP_3T: '3T T1w only', GROUP_3T_FLAIR: '3T T1w + FLAIR'},
)

In [ ]:
fig_7tflair, axes_7tflair, performance_table_7tflair = plot_sensitivity_specificity(
    eval_stats_df,
    analysis_groups=[GROUP_3T, GROUP_7T_ADAPTED, GROUP_7T_ADAPTED_FLAIR],
    harmo_conditions=['harmo'],
    panels=['failed_recons', 'sensitivity', 'sensitivity_mrn', 'specificity', 'mean_no_fp_clusters'],
    group_hue_family={GROUP_3T: "#a9c8f0", GROUP_7T_ADAPTED: '#f4b78a', GROUP_7T_ADAPTED_FLAIR: _with_value('#f4b78a', 0.8)},
    group_tick_labels={GROUP_3T: '3T T1w only', GROUP_7T_ADAPTED: '7T T1w only', GROUP_7T_ADAPTED_FLAIR: '7T T1w + FLAIR'},
)

#plt.show()

### Supplementary Figure 7
MELD-graph detection performance with the basal temporal and frontal cortex masked out of the evaluation

In [ ]:
# the masked cortex on the fsaverage_sym Desikan-Killiany parcellation (lh only: the rh predictions
# and lesion labels are in the xhemi space of the lh)
masked_rois = ['entorhinal', 'fusiform', 'inferiortemporal', 'lateralorbitofrontal',
               'medialorbitofrontal', 'parahippocampal', 'temporalpole']
aparc_labels, _, aparc_names = nib.freesurfer.read_annot(FSAVERAGE_SYM_ANNOT_PATH)
# aparc_labels start at 1000
aparc_labels -= 1000
aparc_names = [name.decode() for name in aparc_names]
roi_mask = np.isin(aparc_labels, [aparc_names.index(name) for name in masked_rois])
# FreeSurferColorLUT index and number of vertices of every masked region
{name: (1000 + aparc_names.index(name), (aparc_labels == aparc_names.index(name)).sum()) for name in masked_rois}


In [ ]:
# patients whose lesion label (projected to fsaverage_sym by vol_eval.py) has any overlap with the masked cortex
# are left out, as are those without a lesion label, for whom this cannot be checked
patients_lesion_in_roi, patients_without_label = [], []
for _, subject in eval_stats_df[eval_stats_df['group'] == 'patient'].drop_duplicates(subset='site_subj_id').iterrows():
    gt_masks = [nib.load(gt_path).get_fdata().squeeze() == 1
                for gt_path in [f'data/results/labeled_predictions/gt_fsaverage_sym/{subject["source"]}'
                                f'/{subject["subject ID"]}/{hemi}.gt.fsaverage_sym.mgh' for hemi in ['lh', 'rh']]
                if os.path.exists(gt_path)]
    if not any(gt_mask.any() for gt_mask in gt_masks):
        patients_without_label.append(subject['site_subj_id'])
    elif any(gt_mask[roi_mask].any() for gt_mask in gt_masks):
        patients_lesion_in_roi.append(subject['site_subj_id'])
print(f'patients with a lesion touching the masked cortex ({len(patients_lesion_in_roi)}): {sorted(patients_lesion_in_roi)}')
print(f'patients without a lesion label on fsaverage_sym ({len(patients_without_label)}): {sorted(patients_without_label)}')


In [ ]:
# we need to merge eval_stats_df onto the cluster_stats_df to get the 
# analysis_group and prediction_path for each cluster, so we can filter out 
# clusters that overlap the masked cortex

# the predicted clusters with the tp / fp status saved to 
# data/results/vol_eval_clusters_<timestamp>.csv (same timestamp as the vol_eval csv)
results_timestamp = re.sub(r'.*vol_eval_(.+)\.csv', r'\1', latest_vol_eval_path())
cluster_stats_df = pd.read_csv(f'data/results/vol_eval_clusters_{results_timestamp}.csv',
                               usecols=['model', 'harmo', 'B0_condition', 'site', 'subject ID', 'cluster', 'cluster_type'])
cluster_stats_df.rename(columns={'site': 'source'}, inplace=True)

# drop the UNION rows from eval_stats_df, which have no clusters of their own
eval_stats_df_masked = eval_stats_df[eval_stats_df['analysis_group'] != GROUP_3T_UNION_7T].copy()
prediction_columns = ['model', 'harmo', 'B0_condition', 'source', 'subject ID']
# merge with the vol_eval stats to get the analysis_group and prediction_path
cluster_stats_df = pd.merge(cluster_stats_df,
                            eval_stats_df_masked[prediction_columns + ['prediction_path', 'analysis_group']],
                            on=prediction_columns, how='inner', validate='many_to_one')

# check whether a cluster overlaps the masked cortex
# cluster indices i are saved on the surface as i or 100*i (most salient vertices)
cluster_stats_df['in_roi'] = False
for prediction_path, clusters in cluster_stats_df.groupby('prediction_path'):
    ids_in_roi = np.unique(np.concatenate([
        nib.load(f'{os.path.dirname(prediction_path)}/fsaverage_sym/{hemi}.prediction.mgh').get_fdata().squeeze()[roi_mask]
        for hemi in ['lh', 'rh']]))
    cluster_stats_df.loc[clusters.index, 'in_roi'] = (clusters['cluster'].isin(ids_in_roi) |
                                                      (100 * clusters['cluster']).isin(ids_in_roi))

# show clusters dropped per B0, condition, tp/fp
cluster_stats_df.groupby(['analysis_group', 'harmo', 'cluster_type'])['in_roi'].agg(dropped='sum', all='size')


In [ ]:
def count_clusters(clusters):
    # tp / fp clusters of every prediction in eval_stats_df_masked; 0 for the predictions without
    # (remaining) clusters, which have no rows in the cluster table
    counts = (clusters.assign(tp=clusters['cluster_type'] == 'tp', fp=clusters['cluster_type'] == 'fp')
              .groupby(prediction_columns)[['tp', 'fp']].sum())
    return eval_stats_df_masked[prediction_columns].join(counts, on=prediction_columns)[['tp', 'fp']].fillna(0)

recon_successful = eval_stats_df_masked['recon_successful'] == 1
patient = eval_stats_df_masked['group'] == 'patient'
control = eval_stats_df_masked['group'] == 'control'

# without masking, the cluster table has to reproduce the cluster counts of vol_eval.py
all_counts = count_clusters(cluster_stats_df)
mismatch = recon_successful & ((all_counts['tp'] != eval_stats_df_masked['tp_clusters']) |
                               (all_counts['fp'] != eval_stats_df_masked['fp_clusters']))
assert not mismatch.any(), ('cluster table does not reproduce tp_clusters / fp_clusters for '
                            f'{eval_stats_df_masked.loc[mismatch, prediction_columns].values.tolist()}')

# the remaining clusters decide tp_patient and tn_control, as at the end of evaluate_prediction() in vol_eval.py
tp_patient_unmasked = eval_stats_df_masked['tp_patient'].copy()
remaining_counts = count_clusters(cluster_stats_df[~cluster_stats_df['in_roi']])
eval_stats_df_masked.loc[recon_successful, 'tp_clusters'] = remaining_counts.loc[recon_successful, 'tp']
eval_stats_df_masked.loc[recon_successful, 'fp_clusters'] = remaining_counts.loc[recon_successful, 'fp']
eval_stats_df_masked.loc[recon_successful & patient, 'tp_patient'] = eval_stats_df_masked['tp_clusters'] > 0
eval_stats_df_masked.loc[recon_successful & control, 'tn_control'] = eval_stats_df_masked['fp_clusters'] == 0

eval_stats_df_masked = eval_stats_df_masked[~eval_stats_df_masked['site_subj_id'].isin(patients_lesion_in_roi +
                                                                                      patients_without_label)]

# remaining patients that are only detected through a cluster touching the masked cortex
flipped = (tp_patient_unmasked == True) & (eval_stats_df_masked['tp_patient'] == False)
eval_stats_df_masked.loc[flipped[flipped].index, ['site_subj_id', 'analysis_group', 'harmo']]


In [ ]:
fig_masked, axes_masked, performance_table_masked = plot_sensitivity_specificity(
    eval_stats_df_masked,
    analysis_groups=[GROUP_3T, GROUP_7T_DEFAULT, GROUP_7T_ADAPTED],
    harmo_conditions=['noharmo', 'harmo'],
    group_hue_family=main_group_hue_family,
    panels=['sensitivity', 'specificity', 'mean_no_fp_clusters'],
    figsize=(6, 5.5),
)

# change Panel A title to "A | Sensitivity (patients without basal frontal or temporal lesion)"
axes_masked[0].set_title("A | Sensitivity (patients without basal frontal or temporal lesion)", 
                         fontsize=10, 
                         ha='left', 
                         x=0.0, 
                         y=1.02, 
                         fontweight='bold')

# change Panel B ylim max to 100
axes_masked[1].set_ylim(0, 100)

In [ ]:
# the masked cortex in red on the fsaverage_sym lh pial surface, seen from lateral, medial and below
os.environ['PYVISTA_OFF_SCREEN'] = 'true'
os.environ['DISPLAY'] = ''  # Disable X server connection attempts
os.environ['EGL_PLATFORM'] = 'surfaceless'  # Use surfaceless EGL platform
pv.start_xvfb()
pv.set_jupyter_backend('static')

with tempfile.TemporaryDirectory(dir='tmp/') as temp_dir:
    cmd = f'apptainer run freesurfer_8.1.0.sif /bin/bash -c "cp \\$FREESURFER_HOME/subjects/fsaverage_sym/surf/lh.pial {temp_dir}"'
    subprocess.run(cmd, shell=True, check=True)
    pial_vertices, pial_faces = nib.freesurfer.io.read_geometry(f'{temp_dir}/lh.pial')

pial_mesh = pv.PolyData(pial_vertices, np.column_stack([[3] * len(pial_faces), pial_faces]).ravel())
pial_mesh['roi_color'] = np.where(roi_mask[:, None], [1.0, 0.0, 0.0], [0.75, 0.75, 0.75])

# camera direction and view-up of each view; the lh lateral surface faces -x, and the ventral view
# has the lateral side up so that anterior is on the left as in the lateral view
views = {'lateral': ([-1, 0, 0], [0, 0, 1]),
         'medial': ([1, 0, 0], [0, 0, 1]),
         'ventral': ([0, 0, -1], [-1, 0, 0])}
center = pial_vertices.mean(axis=0)
distance = 2 * np.ptp(pial_vertices, axis=0).max()

plotter = pv.Plotter(shape=(1, len(views)), window_size=(320 * len(views), 220), border=False, notebook=True)
for i, (direction, view_up) in enumerate(views.values()):
    plotter.subplot(0, i)
    plotter.add_mesh(pial_mesh, scalars='roi_color', rgb=True, smooth_shading=True)
    plotter.camera_position = [center + distance * np.array(direction), center, view_up]
    plotter.enable_parallel_projection()
    plotter.reset_camera()
plotter.show()


### Extra Supplementary Figure
comparison of 7T MPRAGE (Bonn) and 7T MP2RAGE (other sites) performance

In [ ]:
# the 7T T1w sequence is MPRAGE at Bonn and MP2RAGE at all other sites
sequence_site_groups = {'Bonn (7T MPRAGE)': eval_stats_df['site'] == 'Bonn',
                        'Other sites (7T MP2RAGE)': eval_stats_df['site'] != 'Bonn'}
sequence_analysis_groups = [GROUP_7T_DEFAULT, GROUP_7T_ADAPTED]
sequence_harmo_conditions = ['noharmo', 'harmo']
sequence_panels = ['sensitivity', 'specificity', 'mean_no_fp_clusters']

# one column of panels per site group; the left column is lettered A, B, C, the right one D, E, F
fig_sequence = plt.figure(figsize=(11, 7), layout='constrained')
axes_sequence = []
performance_tables_sequence = {}
for column, (subfig, (site_group, in_site_group)) in enumerate(zip(fig_sequence.subfigures(1, 2), sequence_site_groups.items())):
    print(site_group)
    _, axes_site_group, performance_tables_sequence[site_group] = plot_sensitivity_specificity(
        eval_stats_df[in_site_group],
        analysis_groups=sequence_analysis_groups,
        harmo_conditions=sequence_harmo_conditions,
        group_hue_family=main_group_hue_family,
        panels=sequence_panels,
        axes=subfig.subplots(nrows=len(sequence_panels), ncols=1, sharex=True),
        annotate_pairs=False,
    )
    subfig.suptitle(site_group, fontsize=11, fontweight='bold')
    for row, ax in enumerate(axes_site_group):
        ax.set_title(f'{chr(ord("A") + column * len(sequence_panels) + row)}{ax.get_title()[1:]}',
                     fontsize=10, ha='left', x=0.0, y=1.02, fontweight='bold')
    axes_sequence.append(axes_site_group)

# same y-range for the false positive panels of both site groups
fp_row = sequence_panels.index('mean_no_fp_clusters')
fp_ylim_top = max(axes_site_group[fp_row].get_ylim()[1] for axes_site_group in axes_sequence)
for axes_site_group in axes_sequence:
    axes_site_group[fp_row].set_ylim(0, fp_ylim_top)

In [ ]:
# Bonn vs. other sites per harmo condition, analysis group and metric, FDR-corrected over the whole table:
# Fisher's exact test for sensitivity / specificity, Mann-Whitney U test for the false positive clusters
# per subject (not a proportion), on the same subjects as the figure (present in every condition, recon_successful == 1)
fisher_rows = []
for harmo, analysis_group, metric_col in itertools.product(
        sequence_harmo_conditions, sequence_analysis_groups, sequence_panels):
    row = {'harmo': harmo, 'analysis_group': analysis_group, 'metric': metric_col}
    contingency_table, fp_clusters = [], []
    for site_group, in_site_group in sequence_site_groups.items():
        performance_row = performance_tables_sequence[site_group].set_index(['harmo', 'analysis_group']).loc[(harmo, analysis_group)]
        if metric_col == 'mean_no_fp_clusters':
            row[site_group] = f"n={int(performance_row['n_fp_clusters'])}, {performance_row['mean_no_fp_clusters_display']}"
            df_site_group = filter_to_common_subjects(
                eval_stats_df[in_site_group & (eval_stats_df['recon_successful'] == 1)],
                sequence_analysis_groups, sequence_harmo_conditions)
            fp_clusters.append(df_site_group.loc[(df_site_group['harmo'] == harmo) &
                                                 (df_site_group['analysis_group'] == analysis_group), 'fp_clusters'].dropna())
        else:
            k_col, n_col = {'sensitivity': ('k_patients', 'n_patients'),
                            'specificity': ('k_controls', 'n_controls')}[metric_col]
            k, n = performance_row[[k_col, n_col]]
            row[site_group] = f"{int(k)}/{int(n)}, {performance_row[f'{metric_col}_display']}"
            contingency_table.append([k, n - k])
    if metric_col == 'mean_no_fp_clusters':
        row['test'] = 'Mann-Whitney U'
        row['p'] = mannwhitneyu(*fp_clusters).pvalue
    else:
        row['test'] = 'Fisher'
        row['p'] = fisher_exact(contingency_table)[1]
    fisher_rows.append(row)
fisher_table = pd.DataFrame(fisher_rows)
fisher_table['p (FDR)'] = multipletests(fisher_table['p'], method='fdr_bh')[1]
fisher_table